In [1]:
!pip install opencv-python roboflow imageio imageio-ffmpeg -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.3/302.3 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 80.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 119.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 6.1 MB/s eta 0:00:00


In [2]:
from google.colab import drive
import roboflow
import cv2
import os
from pathlib import Path
from tqdm import tqdm
import imageio

In [3]:
def get_extracted_filenames(output_dir: str) -> set:
    if not os.path.exists(output_dir):
        return set()

    existing = {
        f for f in os.listdir(output_dir)
        if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp'))
    }
    print(f"  → {len(existing)} file sudah ada di output_dir")
    return existing

In [4]:
def get_frames_to_extract(total_frames: int, every_n_frames: int, fmt: str) -> dict:
    frames = {}
    extracted_count = 0
    for frame_count in range(total_frames):
        if frame_count % every_n_frames == 0:
            fname = f"frame_{extracted_count:06d}.{fmt}"
            frames[fname] = frame_count
            extracted_count += 1
    return frames

In [5]:
def get_frames_to_skip(all_frames: dict, existing_files: set) -> dict:
    to_extract = {
        fname: idx
        for fname, idx in all_frames.items()
        if fname not in existing_files
    }
    already_exists = len(all_frames) - len(to_extract)

    print(f"\n  Ringkasan:")
    print(f"  Sudah diekstrak : {already_exists} frame")
    print(f"  Belum diekstrak : {len(to_extract)} frame")
    return to_extract

In [6]:
def extract_frames_opencv(video_path: str, output_dir: str, to_extract: dict,
                          fmt: str = 'jpg', quality: int = 95) -> bool:
    os.makedirs(output_dir, exist_ok=True)

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"❌ Error: Tidak bisa membuka video {video_path}")
        return False

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps          = cap.get(cv2.CAP_PROP_FPS)
    width        = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height       = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    print(f"\n📹 Informasi Video:")
    print(f"   Total Frames : {total_frames}")
    print(f"   FPS          : {fps}")
    print(f"   Resolusi     : {width}x{height}")
    print(f"   Durasi       : {total_frames/fps:.2f} detik")

    target_indices = {idx: fname for fname, idx in to_extract.items()}

    extracted, skipped = 0, 0
    frame_count = 0

    with tqdm(total=total_frames, desc="🎬 Extracting") as pbar:
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            if frame_count in target_indices:
                fname    = target_indices[frame_count]
                filepath = os.path.join(output_dir, fname)

                if fmt.lower() == 'png':
                    cv2.imwrite(filepath, frame, [cv2.IMWRITE_PNG_COMPRESSION, 9])
                elif fmt.lower() == 'jpg':
                    cv2.imwrite(filepath, frame, [cv2.IMWRITE_JPEG_QUALITY, quality])

                extracted += 1
            else:
                skipped += 1

            frame_count += 1
            pbar.update(1)

    cap.release()

    print(f"\n✅ Selesai!")
    print(f"   Diekstrak : {extracted} frame")
    print(f"   Dilewati  : {skipped} frame (sudah ada / tidak perlu)")
    print(f"   Output    : {output_dir}")
    return True

In [7]:
def main_extract_frame(video_path: str, output_dir: str, every_n_frames: int,
         fmt: str, quality: int) -> None:

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"❌ Error: Tidak bisa membuka video {video_path}")
        return
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()

    print("📂 Mengecek file yang sudah ada di output_dir...")
    existing_files = get_extracted_filenames(output_dir)

    print("\n🔢 Menghitung frame yang akan diekstrak...")
    all_frames = get_frames_to_extract(total_frames, every_n_frames, fmt)

    to_extract = get_frames_to_skip(all_frames, existing_files)

    if not to_extract:
        print("\n✅ Semua frame sudah diekstrak, tidak ada yang perlu diproses.")
        return

    extract_frames_opencv(video_path, output_dir, to_extract, fmt, quality)

In [8]:
def get_roboflow_filenames(project, batch_name: str) -> tuple[set, int]:
    """
    Return:
        existing_filenames : set nama unik di Roboflow
        total_roboflow     : total foto di Roboflow (termasuk duplikat)
    """
    existing_filenames = set()
    all_names = []
    batch_id = None

    # ── 1. Ambil batch_id ──────────────────────────────────────────────────
    batches = project.get_batches().get("batches", [])
    for batch in batches:
        if batch.get("name") == batch_name:
            batch_id = batch.get("id")
            total_images = batch.get("images", 0)
            break

    if not batch_id:
        print(f"  [ERROR] Batch '{batch_name}' tidak ditemukan.")
        return existing_filenames, 0

    print(f"  → Batch ditemukan: id={batch_id}, total={total_images} foto")

    # ── 2. Ambil semua nama file dengan pagination ─────────────────────────
    offset = 0
    per_page = 100

    while True:
        try:
            results = project.search(
                batch=True,
                batch_id=batch_id,
                offset=offset,
                limit=per_page,
                fields=["name"]
            )

            if isinstance(results, dict):
                results = results.get("results", [])

            if not results:
                break

            for item in results:
                name = item.get("name") or item.get("filename") or ""
                name = os.path.basename(name)
                all_names.append(name)
                existing_filenames.add(name)

            print(f"  [Offset {offset:>5}] → unik: {len(existing_filenames)} | terfetch: {len(all_names)}/{total_images}")
            offset += per_page

        except Exception as e:
            print(f"  [ERROR] Gagal di offset {offset}: {e}")
            break

    # ── 3. Laporan duplikat ────────────────────────────────────────────────
    duplicate_count = len(all_names) - len(existing_filenames)
    print(f"\n  📊 Laporan Roboflow:")
    print(f"     Terfetch total  : {len(all_names)}")
    print(f"     Nama unik       : {len(existing_filenames)}")
    print(f"     Duplikat        : {duplicate_count}")

    return existing_filenames, total_images  # kembalikan total_images dari get_batches

In [9]:
def get_local_filenames(output_dir: str) -> dict:
    local_files = {
        filename: os.path.join(output_dir, filename)
        for filename in os.listdir(output_dir)
        if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp'))
    }
    print(f"  → {len(local_files)} file ditemukan di lokal")
    return local_files

In [10]:
def get_files_to_upload(local_files: dict, existing_filenames: set, total_roboflow: int) -> dict:
    """Kembalikan file lokal yang belum ada di Roboflow."""
    to_upload = {
        filename: path
        for filename, path in local_files.items()
        if filename not in existing_filenames
    }
    already_uploaded = local_files.keys() & existing_filenames

    print(f"\n  📊 Ringkasan:")
    print(f"     File lokal              : {len(local_files)}")
    print(f"     Sudah di Roboflow       : {total_roboflow} (termasuk duplikat)")
    print(f"     Nama unik di Roboflow   : {len(existing_filenames)}")
    print(f"     Match lokal ↔ Roboflow  : {len(already_uploaded)}")
    print(f"     Belum diupload          : {len(to_upload)}")  # seharusnya ~281
    return to_upload

In [11]:
def upload_files_to_roboflow(project, to_upload: dict, batch_name: str) -> None:
    uploaded, failed = 0, 0

    for filename, image_path in to_upload.items():
        print(f"  [UPLOAD] {filename}")
        try:
            project.upload(
                image_path=image_path,
                batch_name=batch_name,
                num_retry_uploads=3
            )
            uploaded += 1
        except Exception as e:
            print(f"  [ERROR]  {filename} → {e}")
            failed += 1

    print(f"\nSelesai → Berhasil: {uploaded} | Gagal: {failed} | Dilewati: {len(to_upload) - uploaded}")

In [12]:
def delete_roboflow_duplicates(project, batch_name: str) -> None:
    """Hapus foto duplikat di batch Roboflow, pertahankan 1 yang pertama ditemukan."""
    batch_id = None

    batches = project.get_batches().get("batches", [])
    for batch in batches:
        if batch.get("name") == batch_name:
            batch_id = batch.get("id")
            total_images = batch.get("images", 0)
            break

    if not batch_id:
        print(f"  [ERROR] Batch '{batch_name}' tidak ditemukan.")
        return

    print(f"  → Batch ditemukan: id={batch_id}, total={total_images} foto")

    seen_names = {}
    duplicate_ids = []
    offset = 0
    per_page = 100

    while True:
        try:
            results = project.search(
                batch=True,
                batch_id=batch_id,
                offset=offset,
                limit=per_page,
                fields=["name"]
            )

            if isinstance(results, dict):
                results = results.get("results", [])

            if not results:
                break

            for item in results:
                name = os.path.basename(item.get("name") or item.get("filename") or "")
                image_id = item.get("id")

                if name in seen_names:
                    duplicate_ids.append(image_id)
                else:
                    seen_names[name] = image_id

            print(f"  [Offset {offset:>5}] → unik: {len(seen_names)} | duplikat ditemukan: {len(duplicate_ids)}")
            offset += per_page

        except Exception as e:
            print(f"  [ERROR] Gagal di offset {offset}: {e}")
            break

    print(f"\n  📊 Laporan Duplikat:")
    print(f"     Total terfetch  : {len(seen_names) + len(duplicate_ids)}")
    print(f"     Nama unik       : {len(seen_names)}")
    print(f"     Akan dihapus    : {len(duplicate_ids)}")

    if not duplicate_ids:
        print("\n  ✅ Tidak ada duplikat, tidak ada yang perlu dihapus.")
        return

    confirm = input(f"\n  ⚠️  Hapus {len(duplicate_ids)} foto duplikat? (ketik 'ya' untuk konfirmasi): ")
    if confirm.strip().lower() != 'ya':
        print("  ❌ Dibatalkan.")
        return

    deleted, failed = 0, 0
    batch_size = 100  # hapus per batch agar tidak overload

    for i in range(0, len(duplicate_ids), batch_size):
        batch_ids = duplicate_ids[i:i + batch_size]
        try:
            project.delete_images(batch_ids)
            deleted += len(batch_ids)
            print(f"  [Hapus {i:>5}~{i+len(batch_ids):>5}] → {deleted}/{len(duplicate_ids)} terhapus...")
        except Exception as e:
            failed += len(batch_ids)
            print(f"  [ERROR] Gagal hapus batch {i}~{i+len(batch_ids)}: {e}")

    print(f"\n  ✅ Selesai!")
    print(f"     Berhasil dihapus : {deleted}")
    print(f"     Gagal            : {failed}")

In [13]:
def main_upload_to_roboflow(project, output_dir: str, batch_name: str) -> None:
    existing_filenames, total_roboflow = get_roboflow_filenames(project, batch_name)
    local_files = get_local_filenames(output_dir)

    to_upload = get_files_to_upload(local_files, existing_filenames, total_roboflow)

    if not to_upload:
        print("\n✅ Semua file sudah ada di Roboflow, tidak ada yang perlu diupload.")
        return

    print("\nMemulai upload...")
    upload_files_to_roboflow(project, to_upload, batch_name)

In [14]:
rf = roboflow.Roboflow(api_key="hRSsLIOXzzSZPkM0Wd54")
project = rf.workspace().project("plastic-trash-detection-p9gqv")
filename = "0412"
MAIN_PATH = "/content/drive/MyDrive/"
# VIDEO_PATH = f"{MAIN_PATH}/DJI_{filename}.MOV" # bisa .MOV atau .MP4
OUTPUT_DIR = f"{MAIN_PATH}/DJISampah/{filename}"
EXTRACT_EVERY_N_FRAMES = 1  # 1 = semua frame, 2 = setiap frame ke-2, dst
OUTPUT_FORMAT = 'jpg'        # 'png' (lossless) atau 'jpg' (lossy)
QUALITY = 98                 # Untuk JPG (1-100)

loading Roboflow workspace...
loading Roboflow project...


In [15]:
# main_extract_frame(VIDEO_PATH, OUTPUT_DIR, EXTRACT_EVERY_N_FRAMES, OUTPUT_FORMAT, QUALITY)

In [16]:
# delete_roboflow_duplicates(project, batch_name=filename)

In [18]:
main_upload_to_roboflow(project, OUTPUT_DIR, filename)

  → Batch ditemukan: id=EwhvrTAHWoH2Xx1hNvR2, total=9334 foto
  [Offset     0] → unik: 100 | terfetch: 100/9334
  [Offset   100] → unik: 200 | terfetch: 200/9334
  [Offset   200] → unik: 300 | terfetch: 300/9334
  [Offset   300] → unik: 400 | terfetch: 400/9334
  [Offset   400] → unik: 500 | terfetch: 500/9334
  [Offset   500] → unik: 589 | terfetch: 600/9334
  [Offset   600] → unik: 680 | terfetch: 700/9334
  [Offset   700] → unik: 774 | terfetch: 800/9334
  [Offset   800] → unik: 868 | terfetch: 900/9334
  [Offset   900] → unik: 954 | terfetch: 1000/9334
  [Offset  1000] → unik: 1032 | terfetch: 1100/9334
  [Offset  1100] → unik: 1132 | terfetch: 1200/9334
  [Offset  1200] → unik: 1232 | terfetch: 1300/9334
  [Offset  1300] → unik: 1328 | terfetch: 1400/9334
  [Offset  1400] → unik: 1428 | terfetch: 1500/9334
  [Offset  1500] → unik: 1510 | terfetch: 1600/9334
  [Offset  1600] → unik: 1609 | terfetch: 1700/9334
  [Offset  1700] → unik: 1709 | terfetch: 1800/9334
  [Offset  1800] → un

KeyboardInterrupt: 